# Experiment 08 — HyDE retrieval with Qwen 2.5 3B Instruct (Colab Free T4)

End-to-end runner for [`docs/plans/hyde_qwen_colab.md`](../docs/plans/hyde_qwen_colab.md).

Pipeline (run cells top-to-bottom):

1. **Phase 0** — Colab setup: T4 GPU, Drive mount, repo clone, install, env vars, Qwen weights snapshot.
2. **Phase 3** — 1-record dry-run + manual GATE (`scripts/exp08_test_one.py`).
3. **Phase 5** — Pilot 5 (`scripts/exp08_run.py --stt 1-5`).
4. **Phase 6** — Full 200 (`scripts/exp08_run.py`).
5. **Phase 7** — Metrics (`scripts/exp08_metrics.py`).
6. **Phase 8** — Funnel (`scripts/exp08_funnel.py`).
7. **Phase 9** — `git add + commit + push exp/08-hyde`.

Secrets (Colab → key icon → New secret): `GITHUB_PAT`, `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`, `NEO4J_DATABASE`.

**Important**: clear ALL cell outputs before committing this notebook back to git — outputs may leak secrets / Neo4j responses.

## Phase 0 — Colab setup

In [ ]:
# 0.1 — Verify T4 GPU. If this prints CPU or fails, switch runtime:
#       Runtime → Change runtime type → T4 GPU.
!nvidia-smi

In [ ]:
# 0.2 — Mount Google Drive (persists across session disconnects).
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 0.3 — Constants. Adjust GITHUB_OWNER + GITHUB_REPO if your fork differs.
REPO_DRIVE_PATH = '/content/drive/MyDrive/legal-graph-kb'
HF_CACHE_PATH   = '/content/drive/MyDrive/hf_cache'
BRANCH          = 'exp/08-hyde'
GITHUB_OWNER    = 'NguyenHuuHoang'   # TODO: replace with your GitHub username
GITHUB_REPO     = 'legal-graph-kb'
import os
os.makedirs(HF_CACHE_PATH, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE_PATH
print('REPO :', REPO_DRIVE_PATH)
print('HF   :', HF_CACHE_PATH)
print('BRANCH:', BRANCH)

In [ ]:
# 0.4 — Clone the repo into Drive on first run; pull + checkout on later runs.
#       Uses GITHUB_PAT from Colab Secrets (fine-grained, repo write scope).
from google.colab import userdata
import os, subprocess

pat = userdata.get('GITHUB_PAT')
remote = f'https://{pat}@github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'

if not os.path.exists(REPO_DRIVE_PATH + '/.git'):
    print('Cloning ...')
    subprocess.run(['git', 'clone', remote, REPO_DRIVE_PATH], check=True)
else:
    print('Repo already on Drive — fetching latest ...')
    subprocess.run(['git', '-C', REPO_DRIVE_PATH, 'fetch', '--all'], check=True)

# Switch to the exp branch (create from main if it doesn't exist yet).
br = subprocess.run(
    ['git', '-C', REPO_DRIVE_PATH, 'rev-parse', '--verify', BRANCH],
    capture_output=True, text=True,
)
if br.returncode != 0:
    print(f'Creating local branch {BRANCH} from origin/main ...')
    subprocess.run(['git', '-C', REPO_DRIVE_PATH, 'checkout', '-B', BRANCH, 'origin/main'], check=True)
else:
    subprocess.run(['git', '-C', REPO_DRIVE_PATH, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DRIVE_PATH, 'pull', 'origin', BRANCH], check=False)

%cd $REPO_DRIVE_PATH
!git status

In [ ]:
# 0.5 — Install project + Colab-only extras.
%pip install -e ".[dev,eval]" --quiet
%pip install --quiet bitsandbytes accelerate

In [ ]:
# 0.6 — Export Neo4j credentials from Colab Secrets so the runner picks them up
#       (the runner does load_dotenv() but Colab has no .env — env vars suffice).
from google.colab import userdata
import os
for key in ('NEO4J_URI', 'NEO4J_USER', 'NEO4J_PASSWORD', 'NEO4J_DATABASE'):
    val = userdata.get(key)
    if val is None:
        raise RuntimeError(f'Missing Colab Secret: {key}')
    os.environ[key] = val
os.environ.setdefault('EMBED_MODEL', 'BAAI/bge-m3')
os.environ.setdefault('EMBED_DEVICE', 'cuda')
os.environ.setdefault('HF_HUB_DISABLE_SYMLINKS_WARNING', '1')
os.environ['HF_HOME'] = HF_CACHE_PATH
print('Neo4j + HF env vars set OK.')

In [ ]:
# 0.7 — Pre-download Qwen weights into Drive so disconnect-resume is fast.
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id='Qwen/Qwen2.5-3B-Instruct',
    cache_dir=HF_CACHE_PATH,
)
print('Qwen weights ready in', HF_CACHE_PATH)

In [ ]:
# 0.8 — Optional idle keepalive (JS). Free tier disconnects on 90 min idle.
from IPython.display import Javascript, display
display(Javascript('''
function KeepAlive(){
  console.log("keepalive ping", new Date());
  document.querySelector("colab-toolbar-button#connect")?.click();
}
setInterval(KeepAlive, 60000);
'''))
print('Idle keepalive armed (60s pings).')

## Phase 3 — 1-record dry-run + manual GATE

Eyeball the hypothetical doc + top-12 delta. Tick the checklist printed at the end of the cell:

- [ ] Hypothetical style matches formal legal text
- [ ] NO 'Điều X', 'Khoản Y', numbered citations
- [ ] NO personal names / dates / specific numbers from the question
- [ ] Gold `L58_2014.A2` still in HyDE top-12
- [ ] No suspiciously off-topic articles in top-3

Any failure → edit `prompts/runtime/hyde_generate.md`, then re-run this cell (HyDE cache key includes prompt_sha, so re-runs invalidate automatically).

In [ ]:
!python scripts/exp08_test_one.py --stt 2 --gold L58_2014.A2

## Phase 4 — Experiment scaffold

Already committed at `experiments/08_hyde_retrieval/{config.yaml, README.md, .gitignore}`. No-op cell — confirm files are present:

In [ ]:
!ls -la experiments/08_hyde_retrieval/

## Phase 5 — Pilot 5

Runs all 4 arms on `stt ∈ 1..5` to verify the cache + pipeline + Neo4j round-trip. Should finish in ~2–4 minutes (Qwen prewarm for 5 questions then 4 retrieval calls each).

In [ ]:
!python scripts/exp08_run.py --stt 1-5 --verbose

## Phase 6 — Full 200

Full dataset. Expected wall time on T4: ~30–35 min first run (HyDE generation dominates). On resume after disconnect, cache hits mean only the remaining records get processed — re-run the same cell.

In [ ]:
!python scripts/exp08_run.py

## Phase 7 — Metrics

Reads records from `experiments/08_hyde_retrieval/results/<arm>/A*.json`, runs gold validation, writes JSON + CSV + Markdown report.

In [ ]:
!python scripts/exp08_metrics.py

## Phase 8 — Funnel (full_rerank_hyde)

Per-stage gold profile for the HyDE-augmented full pipeline. Pairs with [`experiments/06_retrieval_dense_vs_full/report/funnel_full_rerank_K12.md`](../experiments/06_retrieval_dense_vs_full/report/funnel_full_rerank_K12.md) for direct comparison.

In [ ]:
!python scripts/exp08_funnel.py

## Phase 9 — Sync results back to GitHub

Commits the experiment outputs (metrics + report + funnel) to `exp/08-hyde` and pushes. The records under `results/` are gitignored by the experiment's `.gitignore` so they stay local in Drive — keeps the repo lean. User pulls locally afterwards for ADR + changelog work.

**Before running**: ensure all earlier cell outputs are cleared if you plan to also commit this notebook (Cell → All Output → Clear).

In [ ]:
!git add experiments/08_hyde_retrieval/metrics experiments/08_hyde_retrieval/report experiments/08_hyde_retrieval/README.md
!git -c user.email="hoangnh.17@grad.uit.edu.vn" -c user.name="NguyenHuuHoang" commit -m "exp 08 — HyDE results (Qwen 2.5 3B Instruct, full 200)"
!git push origin $BRANCH